In [3]:
"""
================================================================================
NOTEBOOK 1: BioViL-T + FT-Transformer + Lesion-Conditioned Sparse Attention
================================================================================
Title: Multimodal Explainable AI for Early-Stage Skin Cancer Classification
       Using Dermoscopic Images and Clinical Metadata

Architecture:
  - Vision Encoder   : BioViL-T (biomedical vision-language transformer)
  - Metadata Encoder : FT-Transformer (Feature Tokenization Transformer)
  - Fusion           : Lesion-Conditioned Sparse Cross-Attention
  - Task             : Binary classification (Melanoma vs Non-Melanoma)

ASSUMES: Preprocessing already completed.
         Expects: train.csv / val.csv / test.csv + images/ folder per split.

Author  : Medical AI Research Pipeline
Version : 1.0  |  Publication-Quality
================================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & GLOBAL CONFIG
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, math, time, json, random, warnings, zipfile, shutil
from pathlib import Path
from copy import deepcopy
from collections import defaultdict
from functools import lru_cache  # <--- ADD THIS LINE

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, classification_report
)
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms, models
import timm

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ─── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ─── Device ────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {DEVICE}")

# ─── Paths  (adjust to your environment) ───────────────────────────────────
BASE_DIR   = Path("/kaggle/input/datasets/ahmedmohsen2005/final-dataset")
TRAIN_DIR  = BASE_DIR / "train" / "train"
VAL_DIR    = BASE_DIR / "val"   / "val"
TEST_DIR   = BASE_DIR / "test"  / "test"
OUTPUT_DIR = Path("outputs/notebook1_biovil")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Hyperparameters ───────────────────────────────────────────────────────
# ─── Hyperparameters ───────────────────────────────────────────────────────
CFG = dict(
    img_size        = 224,
    batch_size      = 32,      # Increase this if your GPU has 16GB+ VRAM
    num_workers     = 4,       # Set to 4 or os.cpu_count()
    num_epochs      = 1,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    warmup_epochs   = 1,       # Reduced warmup for shorter training
    grad_clip       = 1.0,
    label_smoothing = 0.1,
    ema_decay       = 0.99,
    focal_alpha     = 0.25,
    focal_gamma     = 2.0,
    dropout         = 0.3,
    stoch_depth     = 0.1,
    num_classes     = 2,
    # TURBO SETTING:
    grad_accum      = 2,       # Accumulate gradients over 2 batches before updating
    # FT-Transformer
    ft_d_token      = 128,
    ft_n_heads      = 8,
    ft_n_layers     = 3,
    ft_ffn_factor   = 4/3,
    # Sparse Attention
    sparse_topk     = 8,
    # Vision
    img_embed_dim   = 768,
)

print("[INFO] Config loaded:", CFG)


# ─────────────────────────────────────────────────────────────────────────────
# 1. ZIP EXTRACTION UTILITY
# ─────────────────────────────────────────────────────────────────────────────
def safe_extract_zip(zip_path: Path, extract_to: Path) -> bool:
    """Safely extract a zip archive with validation."""
    if not zip_path.exists():
        print(f"[WARN] ZIP not found: {zip_path}")
        return False
    extract_to.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            bad = zf.testzip()
            if bad:
                print(f"[ERROR] Corrupted entry in ZIP: {bad}")
                return False
            zf.extractall(extract_to)
        print(f"[INFO] Extracted {zip_path.name} → {extract_to}")
        return True
    except Exception as e:
        print(f"[ERROR] Extraction failed: {e}")
        return False


def maybe_extract_all():
    """Extract train/val/test ZIPs if not already extracted."""
    for split in ["train", "val", "test"]:
        zip_file = BASE_DIR / f"{split}.zip"
        target   = BASE_DIR / split
        if not target.exists():
            safe_extract_zip(zip_file, BASE_DIR)


# maybe_extract_all()   # ← uncomment if ZIPs not yet extracted


# ─────────────────────────────────────────────────────────────────────────────
# 2. DATASET
# ─────────────────────────────────────────────────────────────────────────────
class ISICDataset(Dataset):
    IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

    def __init__(self, csv_path: Path, images_dir: Path, metadata_cols: list, transform=None, split: str = "train"):
        self.df = pd.read_csv(csv_path)
        self.images_dir = Path(images_dir)
        self.meta_cols = metadata_cols
        self.transform = transform
        self.split = split

        # FIX 1: Use 'image_fixed' as the lookup key from image_9c63d8.png
        # Ensure we convert to string to handle any numeric filenames
        self.df["__img_path"] = self.df["image_fixed"].astype(str).apply(self._resolve_image)
        
        missing = self.df["__img_path"].isna().sum()
        if missing:
            print(f"[WARN] {missing} images not found in {images_dir} — dropping.")
        
        self.df = self.df[self.df["__img_path"].notna()].reset_index(drop=True)

        # FIX 2: Use 'class' column for labels as seen in image_9c63d8.png
        if len(self.df) > 0:
            self.labels = self.df["class"].values.astype(np.int64)
        else:
            self.labels = np.array([])
            
        print(f"[Dataset:{split}] {len(self.df)} samples found.")

    def _resolve_image(self, image_name):
        # Remove extension if it's already in the CSV string
        clean_name = image_name.replace(".jpg", "").replace(".png", "").replace(".jpeg", "")
        for ext in self.IMAGE_EXTS:
            p = self.images_dir / f"{clean_name}{ext}"
            if p.exists():
                return str(p)
        return None

    def __len__(self):
        return len(self.df)

    @lru_cache(maxsize=2000) # Caches the last 2000 images accessed
    def _load_image(self, path):
        img = Image.open(path).convert("RGB")
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Define the label (Using 'class' column based on your CSV)
        label = int(row["class"]) 

        # 2. Image loading (Turbo cached version)
        try:
            img = self._load_image(row["__img_path"])
        except Exception:
            img = Image.new("RGB", (CFG["img_size"], CFG["img_size"]))

        if self.transform:
            img = self.transform(img)

        # 3. Metadata processing
        meta = torch.tensor(
            row[self.meta_cols].values.astype(np.float32),
            dtype=torch.float32
        )

        return {
            "image"    : img,
            "metadata" : meta,
            "label"    : torch.tensor(label, dtype=torch.long), # Variable 'label' is now defined
            "image_id" : row["image_fixed"], # Consistent with your path resolution
        }


def build_transforms(img_size: int, split: str):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    if split == "train":
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
            transforms.RandomRotation(30),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])


def build_dataloaders(meta_cols: list):
    """Build train/val/test DataLoaders. Adjust CSV/image paths as needed."""
    datasets = {}
    for split, data_dir in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
        csv_p  = data_dir / f"{split}.csv"
        img_p  = data_dir / "images"
        if not csv_p.exists():
            print(f"[WARN] CSV not found: {csv_p} — skipping split.")
            continue
        ds = ISICDataset(
            csv_path     = csv_p,
            images_dir   = img_p,
            metadata_cols= meta_cols,
            transform    = build_transforms(CFG["img_size"], split),
            split        = split,
        )
        datasets[split] = ds

    loaders = {}
    if "train" in datasets:
        labels   = datasets["train"].labels
        counts   = np.bincount(labels)
        w        = 1.0 / counts
        sample_w = torch.tensor([w[l] for l in labels], dtype=torch.double)
        sampler  = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
        loaders["train"] = DataLoader(
    datasets["train"], 
    batch_size=CFG["batch_size"],
    sampler=sampler, 
    num_workers=os.cpu_count(), # Use all available cores
    pin_memory=True, 
    prefetch_factor=2, # Prepare next 2 batches while GPU works
    persistent_workers=True 
)
    for split in ["val", "test"]:
        if split in datasets:
            loaders[split] = DataLoader(
                datasets[split], batch_size=CFG["batch_size"],
                shuffle=False, num_workers=CFG["num_workers"],
                pin_memory=True
            )
    return loaders, datasets


# ─────────────────────────────────────────────────────────────────────────────
# 3. MODEL COMPONENTS
# ─────────────────────────────────────────────────────────────────────────────

# ─── 3.1  Focal Loss ─────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance.
    Reference: Lin et al., 2017 — "Focal Loss for Dense Object Detection"
    alpha : balancing factor for positive class
    gamma : focusing parameter; higher → harder examples weighted more
    """
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.alpha   = alpha
        self.gamma   = gamma
        self.ls      = label_smoothing

    def forward(self, logits, targets):
        # apply label smoothing via soft targets
        n_cls   = logits.size(1)
        soft_t  = torch.full_like(logits, self.ls / (n_cls - 1))
        soft_t.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        log_p   = F.log_softmax(logits, dim=1)
        p       = torch.exp(log_p)
        # focal weight on positive class
        p_t     = (p * soft_t).sum(dim=1)
        fl_w    = self.alpha * (1 - p_t) ** self.gamma
        loss    = -(fl_w * (log_p * soft_t).sum(dim=1)).mean()
        return loss


# ─── 3.2  SAM Optimizer Wrapper ──────────────────────────────────────────────
class SAM(optim.Optimizer):
    """
    Sharpness-Aware Minimization optimizer.
    Reference: Foret et al., 2020
    Finds flatter minima → better generalization.
    """
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None:
                    continue
                e_w = p.grad * scale
                p.add_(e_w)
                self.state[p]["e_w"] = e_w
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                p.sub_(self.state[p]["e_w"])
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                p.grad.norm(p=2).to(shared_device)
                for group in self.param_groups
                for p in group["params"]
                if p.grad is not None
            ]),
            p=2
        )
        return norm

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups


# ─── 3.3  EMA ────────────────────────────────────────────────────────────────
class EMA:
    """Exponential Moving Average of model weights."""
    def __init__(self, model, decay=0.999):
        self.model  = deepcopy(model).eval()
        self.decay  = decay

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data, alpha=1 - self.decay)

    def __call__(self, *args, **kwargs):
        return self.model(*args, **kwargs)


# ─── 3.4  Stochastic Depth ───────────────────────────────────────────────────
class StochasticDepth(nn.Module):
    """Drop entire residual branches during training (DropPath)."""
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        noise = torch.rand(shape, dtype=x.dtype, device=x.device)
        noise.floor_().div_(keep)
        return x * noise


# ─── 3.5  BioViL-T Vision Encoder ────────────────────────────────────────────
class BioViLEncoder(nn.Module):
    """
    BioViL-T-inspired vision encoder for dermoscopic images.
    We use a ViT-B/16 pretrained backbone as the base
    (full BioViL-T weights require Microsoft HuggingFace access;
     swap model_name to 'microsoft/BioViL-T' when weights are available).

    In the medical domain:
    - Self-attention captures long-range spatial dependencies
    - Patch tokens model local lesion regions (borders, pigmentation)
    - [CLS] token represents global lesion semantics
    """
    def __init__(self, embed_dim=768, drop_path=0.1):
        super().__init__()
        self.backbone = timm.create_model(
            "vit_base_patch16_224",   # swap for BioViL-T when available
            pretrained=True,
            num_classes=0,            # remove classification head
        )
        self.drop_path  = StochasticDepth(drop_path)
        self.embed_dim  = embed_dim
        self.proj       = nn.Linear(embed_dim, embed_dim)
        self.norm       = nn.LayerNorm(embed_dim)

    def forward(self, x):
        """
        Returns:
            cls_token  : (B, D)       — global image representation
            patch_tok  : (B, N, D)    — spatial patch tokens for cross-attention
        """
        features = self.backbone.forward_features(x)   # (B, N+1, D)
        cls_token  = features[:, 0]                    # [CLS]
        patch_tok  = features[:, 1:]                   # patch tokens
        cls_token  = self.drop_path(cls_token)
        cls_token  = self.norm(self.proj(cls_token))
        return cls_token, patch_tok


# ─── 3.6  FT-Transformer Metadata Encoder ────────────────────────────────────
class FeatureTokenizer(nn.Module):
    """
    FT-Transformer feature tokenization layer.
    Each metadata feature (numerical or categorical embedding) → a token.
    Reference: Gorishniy et al., 2021 — "Revisiting Deep Learning Models for
               Tabular Data"

    Medical motivation:
    - Age, sex, anatomical site are clinically correlated with melanoma risk
    - Feature tokenization lets the transformer model feature interactions
    - Unlike MLPs, transformers can attend to feature co-occurrences
    """
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.n_features = n_features
        self.d_token    = d_token
        # one embedding weight per feature + CLS token
        self.weight = nn.Parameter(torch.empty(n_features + 1, d_token))
        self.bias   = nn.Parameter(torch.empty(n_features + 1, d_token))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x: torch.Tensor):
        """
        x : (B, n_features)
        Returns tokens : (B, n_features+1, d_token)
        """
        B = x.size(0)
        # numerical features → multiply by per-feature embedding
        feat_tok  = x.unsqueeze(-1) * self.weight[:-1]       # (B, F, D)
        feat_tok  = feat_tok + self.bias[:-1]
        # [CLS] token for global metadata summary
        cls_tok   = self.weight[-1].unsqueeze(0).expand(B, 1, -1) + self.bias[-1]
        tokens    = torch.cat([cls_tok, feat_tok], dim=1)    # (B, F+1, D)
        return tokens


class FTTransformerBlock(nn.Module):
    """Single FT-Transformer encoder block with pre-LN."""
    def __init__(self, d_token: int, n_heads: int, ffn_factor=4/3, dropout=0.1, drop_path=0.0):
        super().__init__()
        self.norm1  = nn.LayerNorm(d_token)
        self.attn   = nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
        self.norm2  = nn.LayerNorm(d_token)
        d_ff        = int(d_token * ffn_factor * 4)    # FT paper uses 4/3 * 4*d
        self.ff     = nn.Sequential(
            nn.Linear(d_token, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_token), nn.Dropout(dropout)
        )
        self.drop_path = StochasticDepth(drop_path)

    def forward(self, x):
        z, _  = self.attn(self.norm1(x), self.norm1(x), self.norm1(x))
        x     = x + self.drop_path(z)
        x     = x + self.drop_path(self.ff(self.norm2(x)))
        return x


class FTTransformerEncoder(nn.Module):
    """
    Full FT-Transformer encoder for clinical metadata.
    Captures non-linear feature interactions crucial for melanoma prediction.
    """
    def __init__(self, n_features, d_token=128, n_heads=8, n_layers=3,
                 ffn_factor=4/3, dropout=0.1, drop_path=0.1):
        super().__init__()
        self.tokenizer  = FeatureTokenizer(n_features, d_token)
        self.blocks     = nn.ModuleList([
            FTTransformerBlock(d_token, n_heads, ffn_factor, dropout, drop_path)
            for _ in range(n_layers)
        ])
        self.norm       = nn.LayerNorm(d_token)
        self.d_token    = d_token

    def forward(self, x):
        """
        x       : (B, n_features)
        Returns : cls_embed (B, d_token), all_tokens (B, n_features+1, d_token)
        """
        tokens = self.tokenizer(x)
        for blk in self.blocks:
            tokens = blk(tokens)
        tokens      = self.norm(tokens)
        cls_embed   = tokens[:, 0]          # [CLS] = global metadata summary
        feat_tokens = tokens[:, 1:]         # per-feature tokens
        return cls_embed, feat_tokens


# ─── 3.7  Lesion-Conditioned Sparse Cross-Attention ──────────────────────────
class LesionConditionedSparseAttention(nn.Module):
    """
    Lesion-Conditioned Sparse Cross-Attention (LCSA) fusion module.

    Medical Motivation:
    ─────────────────────────────────────────────────────────────────
    Dermatologists do not uniformly weight all image regions.
    They focus attention on regions CONDITIONED on clinical context:
    - For elderly male patients → attend more to acral regions
    - For history-of-MM patients → attend more to lesion boundaries
    - Dermoscopic type → guides which visual features matter

    LCSA implements this clinically-inspired reasoning:
    1. Metadata context (from FT-Transformer) conditions image attention
    2. Sparse top-k selection mimics expert focus on salient regions
    3. Cross-attention lets metadata query visual patch features
    4. Bidirectional: image tokens also attend to metadata features

    Architecture:
    ─────────────────────────────────────────────────────────────────
    [Metadata CLS] ──→ Query projector ──→ Q
    [Image patches] ──→ Key/Value proj  ──→ K, V
    Sparse top-k attention mask (retain top-k patches per head)
    Output: fused representation ∈ ℝ^(B × D)
    """

    def __init__(self, img_dim=768, meta_dim=128, fusion_dim=256,
                 n_heads=8, topk=8, dropout=0.1):
        super().__init__()
        assert fusion_dim % n_heads == 0
        self.n_heads    = n_heads
        self.head_dim   = fusion_dim // n_heads
        self.topk       = topk
        self.scale      = self.head_dim ** -0.5

        # project both modalities to fusion_dim
        self.img_proj   = nn.Linear(img_dim,  fusion_dim)
        self.meta_proj  = nn.Linear(meta_dim, fusion_dim)

        # query/key/value projectors
        self.q_proj     = nn.Linear(fusion_dim, fusion_dim)   # meta → Q
        self.k_proj     = nn.Linear(fusion_dim, fusion_dim)   # img  → K
        self.v_proj     = nn.Linear(fusion_dim, fusion_dim)   # img  → V

        # reverse direction: image queries metadata
        self.q2_proj    = nn.Linear(fusion_dim, fusion_dim)
        self.k2_proj    = nn.Linear(fusion_dim, fusion_dim)
        self.v2_proj    = nn.Linear(fusion_dim, fusion_dim)

        self.out_proj   = nn.Linear(fusion_dim * 2, fusion_dim)
        self.norm       = nn.LayerNorm(fusion_dim)
        self.dropout    = nn.Dropout(dropout)

        # gating — learned balance between image and metadata
        self.gate       = nn.Sequential(
            nn.Linear(fusion_dim * 2, 2), nn.Softmax(dim=-1)
        )

    def _sparse_attention(self, q, k, v, topk):
        """
        Compute attention with top-k sparse selection.
        q: (B, Hq, 1, Dh)    — single query per sample (CLS)
        k: (B, H,  N, Dh)    — N patch keys
        v: (B, H,  N, Dh)    — N patch values
        """
        scores  = (q @ k.transpose(-2, -1)) * self.scale  # (B, H, 1, N)
        # ── sparse mask: keep only top-k patches ──────────────────────
        topk_v  = min(topk, scores.size(-1))
        top_val, top_idx = scores.topk(topk_v, dim=-1)    # (B, H, 1, k)
        mask    = torch.full_like(scores, float("-inf"))
        mask.scatter_(-1, top_idx, top_val)
        weights = F.softmax(mask, dim=-1)
        weights = self.dropout(weights)
        out     = (weights @ v).squeeze(2)                 # (B, H, Dh)
        # store attention weights for XAI
        self._last_attn_weights = weights.detach()
        return out

    def forward(self, img_cls, patch_tokens, meta_cls, feat_tokens):
        """
        img_cls     : (B, img_dim)        — global image embedding
        patch_tokens: (B, N, img_dim)     — spatial image patches
        meta_cls    : (B, meta_dim)       — global metadata embedding
        feat_tokens : (B, F, meta_dim)    — per-feature metadata tokens

        Returns:
            fused   : (B, fusion_dim)
            gate_w  : (B, 2)              — modality gate weights (for XAI)
        """
        B = img_cls.size(0)

        # ── project to fusion space ────────────────────────────────────
        img_f   = self.img_proj(patch_tokens)               # (B, N, D)
        meta_f  = self.meta_proj(feat_tokens)               # (B, F, D)
        img_c   = self.img_proj(img_cls).unsqueeze(1)       # (B, 1, D)
        meta_c  = self.meta_proj(meta_cls).unsqueeze(1)     # (B, 1, D)

        def reshape_heads(t, n_heads, head_dim):
            B_, S, D = t.shape
            return t.view(B_, S, n_heads, head_dim).permute(0, 2, 1, 3)

        # ── Direction 1: metadata queries image patches ─────────────────
        Q  = reshape_heads(self.q_proj(meta_c), self.n_heads, self.head_dim)
        K  = reshape_heads(self.k_proj(img_f),  self.n_heads, self.head_dim)
        V  = reshape_heads(self.v_proj(img_f),  self.n_heads, self.head_dim)
        out1 = self._sparse_attention(Q, K, V, self.topk)      # (B, H, Dh)
        out1 = out1.reshape(B, -1)                              # (B, D)
        self._attn1 = self._last_attn_weights

        # ── Direction 2: image queries metadata features ─────────────────
        Q2 = reshape_heads(self.q2_proj(img_c),    self.n_heads, self.head_dim)
        K2 = reshape_heads(self.k2_proj(meta_f),   self.n_heads, self.head_dim)
        V2 = reshape_heads(self.v2_proj(meta_f),   self.n_heads, self.head_dim)
        out2 = self._sparse_attention(Q2, K2, V2, meta_f.size(1))
        out2 = out2.reshape(B, -1)                              # (B, D)
        self._attn2 = self._last_attn_weights

        # ── gated fusion ─────────────────────────────────────────────────
        combined = torch.cat([out1, out2], dim=-1)              # (B, 2D)
        gate_w   = self.gate(combined)                          # (B, 2)
        fused    = gate_w[:, 0:1] * out1 + gate_w[:, 1:2] * out2
        fused    = self.norm(self.out_proj(combined))
        return fused, gate_w


# ─── 3.8  Full BioViL + FT-Transformer Multimodal Model ──────────────────────
class BioViLFTModel(nn.Module):
    """
    Full Multimodal Model:
        BioViL-T Vision Encoder
      + FT-Transformer Metadata Encoder
      + Lesion-Conditioned Sparse Cross-Attention Fusion
      → Binary Classifier

    Publication Reference Architecture:
    ────────────────────────────────────────────────────────────────────
    Image  ──[ViT patches]──→ BioViL-T Encoder ──→ CLS, {p_i}
    Meta   ──[feature tok]──→ FT-Transformer   ──→ CLS_m, {f_j}
                               ↓
              Lesion-Conditioned Sparse Cross-Attention
                   (meta queries image, image queries meta)
                               ↓
              Gated Fusion → MLP Head → P(Melanoma)
    """

    def __init__(self, n_meta_features: int, n_classes=2):
        super().__init__()
        self.vision_enc = BioViLEncoder(
            embed_dim  = CFG["img_embed_dim"],
            drop_path  = CFG["stoch_depth"]
        )
        self.meta_enc   = FTTransformerEncoder(
            n_features = n_meta_features,
            d_token    = CFG["ft_d_token"],
            n_heads    = CFG["ft_n_heads"],
            n_layers   = CFG["ft_n_layers"],
            ffn_factor = CFG["ft_ffn_factor"],
            dropout    = CFG["dropout"],
            drop_path  = CFG["stoch_depth"],
        )
        self.fusion     = LesionConditionedSparseAttention(
            img_dim    = CFG["img_embed_dim"],
            meta_dim   = CFG["ft_d_token"],
            fusion_dim = 256,
            n_heads    = 8,
            topk       = CFG["sparse_topk"],
            dropout    = CFG["dropout"],
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(256),
            nn.Dropout(CFG["dropout"]),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(CFG["dropout"] / 2),
            nn.Linear(128, n_classes),
        )

    def forward(self, image, metadata, return_features=False):
        img_cls, patch_tok   = self.vision_enc(image)
        meta_cls, feat_tok   = self.meta_enc(metadata)
        fused, gate_w        = self.fusion(img_cls, patch_tok, meta_cls, feat_tok)
        logits               = self.classifier(fused)
        if return_features:
            return logits, fused, gate_w
        return logits


# ─────────────────────────────────────────────────────────────────────────────
# 4. TRAINING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def cosine_schedule_with_warmup(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4, mode="max"):
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.best       = None
        self.counter    = 0
        self.stop       = False

    def __call__(self, metric):
        if self.best is None:
            self.best = metric
            return False
        improved = (metric > self.best + self.min_delta if self.mode == "max"
                    else metric < self.best - self.min_delta)
        if improved:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def train_one_epoch(model, loader, optimizer, criterion, scaler, ema, device):
    model.train()
    total_loss, n = 0.0, 0
    optimizer.zero_grad() # Reset at start
    
    for batch_idx, batch in enumerate(loader):
        imgs = batch["image"].to(device, non_blocking=True) # non_blocking for speed
        meta = batch["metadata"].to(device, non_blocking=True)
        lbls = batch["label"].to(device, non_blocking=True)

        # ─── Pass 1: Standard Forward/Backward ───
        with autocast(enabled=True, dtype=torch.float16):
            logits = model(imgs, meta)
            loss = criterion(logits, lbls) / CFG["grad_accum"] # Scale for accumulation
        
        scaler.scale(loss).backward()
        
        # Only perform the expensive SAM steps after accumulating enough gradients
        if (batch_idx + 1) % CFG["grad_accum"] == 0 or (batch_idx + 1) == len(loader):
            # SAM First Step: Unscale and move weights
            scaler.unscale_(optimizer)
            optimizer.first_step(zero_grad=True)

            # ─── Pass 2: Perturbed Forward/Backward ───
            with autocast(enabled=True, dtype=torch.float16):
                loss2 = criterion(model(imgs, meta), lbls) / CFG["grad_accum"]
            
            scaler.scale(loss2).backward()
            
            # SAM Second Step: Update original weights
            optimizer.second_step(zero_grad=True)
            
            scaler.update()
            ema.update(model)
            optimizer.zero_grad() # Prepare for next accumulation cycle

        total_loss += loss.item() * imgs.size(0) * CFG["grad_accum"]
        n += imgs.size(0)

    return total_loss / n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    all_logits, all_labels = [], []
    total_loss, n = 0.0, 0
    for batch in loader:
        imgs  = batch["image"].to(device)
        meta  = batch["metadata"].to(device)
        lbls  = batch["label"].to(device)
        with autocast():
            logits = model(imgs, meta)
            loss   = criterion(logits, lbls)
        total_loss += loss.item() * imgs.size(0)
        n           += imgs.size(0)
        all_logits.append(logits.cpu().float())
        all_labels.append(lbls.cpu())
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels).numpy()
    probs  = torch.softmax(logits, dim=1)[:, 1].numpy()
    preds  = logits.argmax(1).numpy()
    metrics = compute_metrics(labels, preds, probs)
    metrics["loss"] = total_loss / n
    return metrics


def compute_metrics(labels, preds, probs):
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return dict(
        accuracy    = accuracy_score(labels, preds),
        precision   = precision_score(labels, preds, zero_division=0),
        recall      = recall_score(labels, preds, zero_division=0),
        f1          = f1_score(labels, preds, zero_division=0),
        roc_auc     = roc_auc_score(labels, probs),
        pr_auc      = average_precision_score(labels, probs),
        sensitivity = tp / (tp + fn + 1e-8),
        specificity = tn / (tn + fp + 1e-8),
        ppv         = tp / (tp + fp + 1e-8),
        npv         = tn / (tn + fn + 1e-8),
    )


def train_pipeline(model, loaders, n_epochs=None):
    n_epochs  = n_epochs or CFG["num_epochs"]
    criterion = FocalLoss(
        alpha=CFG["focal_alpha"],
        gamma=CFG["focal_gamma"],
        label_smoothing=CFG["label_smoothing"]
    ).to(DEVICE)

    base_opt  = optim.AdamW
    optimizer = SAM(
        model.parameters(), base_opt,
        lr=CFG["lr"], weight_decay=CFG["weight_decay"]
    )
    scheduler = cosine_schedule_with_warmup(
        optimizer.base_optimizer,
        CFG["warmup_epochs"], n_epochs
    )
    scaler    = GradScaler()
    ema       = EMA(model, CFG["ema_decay"])
    early_s   = EarlyStopping(patience=10, mode="max")

    history   = defaultdict(list)
    best_auc  = 0.0
    best_ckpt = OUTPUT_DIR / "best_model.pth"

    for epoch in range(1, n_epochs + 1):
        t0        = time.time()
        train_loss = train_one_epoch(model, loaders["train"], optimizer,
                                     criterion, scaler, ema, DEVICE)
        val_met   = evaluate(ema.model, loaders["val"], criterion, DEVICE)
        scheduler.step()

        for k, v in val_met.items():
            history[f"val_{k}"].append(v)
        history["train_loss"].append(train_loss)

        if val_met["roc_auc"] > best_auc:
            best_auc = val_met["roc_auc"]
            torch.save({
                "epoch"      : epoch,
                "model"      : model.state_dict(),
                "ema"        : ema.model.state_dict(),
                "optimizer"  : optimizer.base_optimizer.state_dict(),
                "best_auc"   : best_auc,
            }, best_ckpt)

        elapsed = time.time() - t0
        print(f"[Epoch {epoch:03d}/{n_epochs}] "
              f"train_loss={train_loss:.4f} | "
              f"val_auc={val_met['roc_auc']:.4f} | "
              f"val_f1={val_met['f1']:.4f} | "
              f"val_acc={val_met['accuracy']:.4f} | "
              f"{elapsed:.1f}s")

        if early_s(val_met["roc_auc"]):
            print("[INFO] Early stopping triggered.")
            break

    # restore best weights
    ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)   
    ema.model.load_state_dict(ckpt["ema"])
    print(f"\n[INFO] Best Val AUC: {best_auc:.4f}")
    return history


# ─────────────────────────────────────────────────────────────────────────────
# 5. EVALUATION & REPORTING
# ─────────────────────────────────────────────────────────────────────────────
def full_evaluation(model, loader, split_name="test"):
    model.eval()
    all_logits, all_labels, all_probs = [], [], []
    all_features, all_gates = [], []
    all_ids = []

    with torch.no_grad():
        for batch in loader:
            imgs  = batch["image"].to(DEVICE)
            meta  = batch["metadata"].to(DEVICE)
            lbls  = batch["label"].numpy()
            logits, feat, gate = model(imgs, meta, return_features=True)
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_logits.append(logits.cpu())
            all_labels.append(lbls)
            all_probs.append(probs.cpu().numpy())
            all_features.append(feat.cpu().numpy())
            all_gates.append(gate.cpu().numpy())
            all_ids.extend(batch["image"])

    labels   = np.concatenate(all_labels)
    probs    = np.concatenate(all_probs)
    preds    = (probs >= 0.5).astype(int)
    features = np.concatenate(all_features)
    gates    = np.concatenate(all_gates)

    metrics  = compute_metrics(labels, preds, probs)

    print(f"\n{'='*60}")
    print(f"  {split_name.upper()} EVALUATION — BioViL-T + FT-Transformer")
    print(f"{'='*60}")
    for k, v in metrics.items():
        print(f"  {k:15s}: {v:.4f}")
    print(f"{'='*60}\n")

    cm = confusion_matrix(labels, preds)
    print(classification_report(labels, preds, target_names=["Non-Mel", "Melanoma"]))

    return {
        "labels"  : labels,
        "probs"   : probs,
        "preds"   : preds,
        "features": features,
        "gates"   : gates,
        "ids"     : all_ids,
        "metrics" : metrics,
    }


# ─────────────────────────────────────────────────────────────────────────────
# 6. XAI — IMAGE EXPLANATIONS
# ─────────────────────────────────────────────────────────────────────────────

class GradCAM:
    """
    Gradient-weighted Class Activation Mapping.
    Reference: Selvaraju et al., 2017

    For dermoscopic images: highlights spatial regions (borders,
    pigmentation patches) most influential for the prediction.
    """
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model      = model
        self.gradients  = None
        self.activations= None
        self._hooks     = []
        self._hooks.append(
            target_layer.register_forward_hook(self._save_activation)
        )
        self._hooks.append(
            target_layer.register_full_backward_hook(self._save_gradient)
        )

    def _save_activation(self, m, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, m, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, image, metadata, class_idx=None):
        self.model.eval()
        image    = image.unsqueeze(0).to(DEVICE).requires_grad_(True)
        metadata = metadata.unsqueeze(0).to(DEVICE)
        logits   = self.model(image, metadata)
        if class_idx is None:
            class_idx = logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, class_idx].backward()

        grads   = self.gradients  # (1, C, H, W)  or (1, N, D) for ViT
        acts    = self.activations

        # ViT patch tokens → reshape to spatial
        if grads.ndim == 3:   # (B, N, D)
            B, N, D  = grads.shape
            side     = int(math.sqrt(N))
            grads    = grads[:, 1:]  # remove CLS
            acts     = acts[:, 1:]
            grads    = grads.mean(dim=-1, keepdim=True)
            cam      = (grads * acts).sum(dim=-1).squeeze()   # (N,)
            cam      = cam.reshape(side, side)
        else:
            weights  = grads.mean(dim=(2, 3), keepdim=True)
            cam      = (weights * acts).sum(dim=1).squeeze()

        cam      = F.relu(cam)
        cam      = cam.cpu().numpy()
        cam      = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        self.remove_hooks()
        return cam

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


class GradCAMPlusPlus(GradCAM):
    """
    Grad-CAM++ — improved class activation mapping with second-order gradients.
    Better at multi-instance localization (multiple lesion foci).
    """
    def __call__(self, image, metadata, class_idx=None):
        self.model.eval()
        image    = image.unsqueeze(0).to(DEVICE).requires_grad_(True)
        metadata = metadata.unsqueeze(0).to(DEVICE)
        logits   = self.model(image, metadata)
        if class_idx is None:
            class_idx = logits.argmax(1).item()
        self.model.zero_grad()
        logits[0, class_idx].backward()

        grads = self.gradients
        acts  = self.activations
        if grads.ndim == 3:
            return super().__call__(image.squeeze(0).detach(), metadata.squeeze(0))

        eps  = 1e-8
        g2   = grads ** 2
        g3   = grads ** 3
        denom= 2 * g2 + (acts * g3).sum(dim=(2, 3), keepdim=True)
        alpha= g2 / (denom + eps)
        w    = (alpha * F.relu(logits[0, class_idx].detach() * grads)).sum(dim=(2,3), keepdim=True)
        cam  = (w * acts).sum(dim=1).squeeze()
        cam  = F.relu(cam).cpu().numpy()
        cam  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        self.remove_hooks()
        return cam


class ScoreCAM:
    """
    Score-CAM: gradient-free CAM using activation-masked forward passes.
    More stable than gradient methods for complex multimodal inputs.
    """
    def __init__(self, model: nn.Module, target_layer: nn.Module, n_acts=20):
        self.model       = model
        self.target_layer= target_layer
        self.n_acts      = n_acts
        self.activations = None
        self._hook       = target_layer.register_forward_hook(self._save)

    def _save(self, m, inp, out):
        self.activations = out.detach()

    @torch.no_grad()
    def __call__(self, image, metadata, class_idx=None):
        self.model.eval()
        img_t  = image.unsqueeze(0).to(DEVICE)
        meta_t = metadata.unsqueeze(0).to(DEVICE)
        base_logits = self.model(img_t, meta_t)
        if class_idx is None:
            class_idx = base_logits.argmax(1).item()

        # run forward to get activations
        _ = self.model(img_t, meta_t)
        acts = self.activations.clone()

        if acts.ndim == 3:   # ViT: (B, N, D) — use mean of last D dims
            B, N, D = acts.shape
            acts    = acts[:, 1:].mean(dim=-1)  # (B, N-1)
            side    = int(math.sqrt(acts.size(-1)))
            act_map = acts.reshape(1, side, side)
        else:
            act_map = acts[0]    # (C, H, W)

        H = W = CFG["img_size"]
        scores  = []
        n_chans = min(self.n_acts, act_map.size(0))
        for i in range(n_chans):
            chan = act_map[i].unsqueeze(0).unsqueeze(0)
            m    = F.interpolate(chan, (H, W), mode="bilinear", align_corners=False)
            m    = (m - m.min()) / (m.max() - m.min() + 1e-8)
            masked_img = img_t * m
            with torch.no_grad():
                score = torch.softmax(self.model(masked_img, meta_t), dim=1)[0, class_idx]
            scores.append(score.item())

        scores = torch.tensor(scores)
        weights= F.softmax(scores, dim=0)
        cam    = torch.zeros(H, W)
        for i, w in enumerate(weights):
            chan = act_map[i % n_chans].unsqueeze(0).unsqueeze(0)
            m    = F.interpolate(chan.float(), (H, W), mode="bilinear", align_corners=False).squeeze()
            cam += w * m.cpu()

        cam = F.relu(cam).numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        self._hook.remove()
        return cam


class IntegratedGradients:
    """
    Integrated Gradients for pixel-level attribution.
    Reference: Sundararajan et al., 2017
    Satisfies completeness axiom: sum of attributions = model output difference.
    """
    def __init__(self, model: nn.Module, n_steps=50):
        self.model   = model
        self.n_steps = n_steps

    def __call__(self, image, metadata, baseline=None, class_idx=None):
        img_t  = image.unsqueeze(0).to(DEVICE)
        meta_t = metadata.unsqueeze(0).to(DEVICE)
        if baseline is None:
            baseline = torch.zeros_like(img_t)

        # linear interpolation baseline → input
        alphas     = torch.linspace(0, 1, self.n_steps, device=DEVICE)
        int_grads  = torch.zeros_like(img_t)

        for alpha in alphas:
            inp = (baseline + alpha * (img_t - baseline)).requires_grad_(True)
            logits = self.model(inp, meta_t)
            if class_idx is None:
                class_idx = logits.argmax(1).item()
            self.model.zero_grad()
            logits[0, class_idx].backward()
            int_grads += inp.grad.detach()

        ig = (img_t - baseline) * int_grads / self.n_steps
        ig = ig.squeeze().abs().mean(dim=0).cpu().numpy()
        ig = (ig - ig.min()) / (ig.max() - ig.min() + 1e-8)
        return ig


class OcclusionSensitivity:
    """
    Occlusion Sensitivity: systematically occlude patches and measure score drop.
    Medical relevance: identifies clinically critical lesion sub-regions.
    """
    def __init__(self, model, patch_size=32, stride=16):
        self.model      = model
        self.patch_size = patch_size
        self.stride     = stride

    @torch.no_grad()
    def __call__(self, image, metadata, class_idx=None):
        H = W = CFG["img_size"]
        img_t  = image.unsqueeze(0).to(DEVICE)
        meta_t = metadata.unsqueeze(0).to(DEVICE)
        base   = torch.softmax(self.model(img_t, meta_t), dim=1)
        if class_idx is None:
            class_idx = base.argmax(1).item()
        base_score = base[0, class_idx].item()

        heatmap = np.zeros((H, W))
        counts  = np.zeros((H, W)) + 1e-8
        ps, st  = self.patch_size, self.stride

        for y in range(0, H - ps + 1, st):
            for x in range(0, W - ps + 1, st):
                occ = img_t.clone()
                occ[:, :, y:y+ps, x:x+ps] = 0.0
                score = torch.softmax(self.model(occ, meta_t), dim=1)[0, class_idx].item()
                drop  = base_score - score
                heatmap[y:y+ps, x:x+ps] += drop
                counts[y:y+ps, x:x+ps]  += 1

        heatmap = heatmap / counts
        heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
        return heatmap


class SmoothGrad:
    """
    SmoothGrad: averages gradients over noisy input copies.
    Reduces gradient noise for sharper, more reliable saliency maps.
    """
    def __init__(self, model, n_samples=25, noise_std=0.1):
        self.model     = model
        self.n_samples = n_samples
        self.noise_std = noise_std

    def __call__(self, image, metadata, class_idx=None):
        img_t  = image.unsqueeze(0).to(DEVICE)
        meta_t = metadata.unsqueeze(0).to(DEVICE)
        smooth = torch.zeros_like(img_t)

        for _ in range(self.n_samples):
            noise  = torch.randn_like(img_t) * self.noise_std
            inp    = (img_t + noise).requires_grad_(True)
            logits = self.model(inp, meta_t)
            if class_idx is None:
                class_idx = logits.argmax(1).item()
            self.model.zero_grad()
            logits[0, class_idx].backward()
            smooth += inp.grad.detach().abs()

        sg = (smooth / self.n_samples).squeeze().mean(dim=0).cpu().numpy()
        sg = (sg - sg.min()) / (sg.max() - sg.min() + 1e-8)
        return sg


class AttentionRollout:
    """
    Attention Rollout for Vision Transformers.
    Reference: Abnar & Zuidema, 2020
    NOTE: Attention rollout provides INTERPRETABILITY SIGNALS about which
    image regions the transformer processes, NOT proof of causal importance.
    """
    def __init__(self, model: nn.Module):
        self.model    = model
        self.attns    = []
        self._hooks   = []
        for blk in model.vision_enc.backbone.blocks:
            self._hooks.append(
                blk.attn.register_forward_hook(self._save_attn)
            )

    def _save_attn(self, m, inp, out):
        # for timm ViT blocks, attn output includes weights via attn.softmax
        # we capture activation output and derive attention
        pass   # will use direct attn access

    @torch.no_grad()
    def __call__(self, image, metadata):
        self.attns = []
        img_t  = image.unsqueeze(0).to(DEVICE)
        meta_t = metadata.unsqueeze(0).to(DEVICE)

        # collect attention matrices by patching forward
        attn_maps = []
        x = self.model.vision_enc.backbone.patch_embed(img_t)
        x = self.model.vision_enc.backbone._pos_embed(x)
        for blk in self.model.vision_enc.backbone.blocks:
            # manually run attention to capture weights
            B, N, C = x.shape
            qkv   = blk.attn.qkv(blk.norm1(x)).reshape(B, N, 3, blk.attn.num_heads, -1).permute(2,0,3,1,4)
            q, k, _ = qkv.unbind(0)
            scale = blk.attn.scale
            attn  = (q @ k.transpose(-2,-1)) * scale
            attn  = attn.softmax(dim=-1)
            attn_maps.append(attn.mean(dim=1).cpu())  # (B, N, N)
            x     = blk(x)

        # rollout
        result = torch.eye(attn_maps[0].size(-1))
        for a in attn_maps:
            a = a.squeeze(0)
            a = a + torch.eye(a.size(-1))
            a = a / a.sum(dim=-1, keepdim=True)
            result = a @ result

        # CLS → patches
        mask = result[0, 1:]   # (N-1,)
        side = int(math.sqrt(mask.size(0)))
        mask = mask.reshape(side, side).numpy()
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        for h in self._hooks:
            h.remove()
        return mask


def overlay_heatmap(img_tensor, heatmap, alpha=0.5, colormap=plt.cm.jet):
    """Overlay a heatmap on a denormalized image tensor."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = img_tensor.permute(1,2,0).cpu().numpy()
    img  = (img * std + mean).clip(0, 1)
    h, w = img.shape[:2]
    heat = np.array(Image.fromarray(
        (heatmap * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR)
    ) / 255.0
    colored = colormap(heat)[:, :, :3]
    overlay = alpha * colored + (1 - alpha) * img
    return overlay.clip(0, 1)


# ─────────────────────────────────────────────────────────────────────────────
# 7. XAI — TABULAR / METADATA
# ─────────────────────────────────────────────────────────────────────────────

def compute_shap_values(model, loader, meta_cols, n_bg=20, n_exp=100):
    """
    SHAP DeepExplainer for metadata features.
    Background: mean metadata from training set (representative baseline).
    """
    try:
        import shap
    except ImportError:
        print("[WARN] shap not installed. pip install shap")
        return None, None

    model.eval()

    # collect background metadata
    bg_metas = []
    for batch in loader:
        bg_metas.append(batch["metadata"])
        if sum(b.size(0) for b in bg_metas) >= n_bg:
            break
    bg = torch.cat(bg_metas)[:n_bg].to(DEVICE)

    # wrapper: metadata-only forward through metadata encoder + classifier
    class MetaOnlyWrapper(nn.Module):
        def __init__(self, full_model):
            super().__init__()
            self.m = full_model
            # use a fixed dummy image
            self._dummy_img = torch.zeros(
                1, 3, CFG["img_size"], CFG["img_size"], device=DEVICE
            )
        def forward(self, meta):
            B = meta.size(0)
            img = self._dummy_img.expand(B, -1, -1, -1)
            return self.m(img, meta)

    wrapper  = MetaOnlyWrapper(model)
    explainer= shap.DeepExplainer(wrapper, bg)

    # explain test samples
    test_metas = []
    for batch in loader:
        test_metas.append(batch["metadata"])
        if sum(b.size(0) for b in test_metas) >= n_exp:
            break
    test_m   = torch.cat(test_metas)[:n_exp].to(DEVICE)
    shap_vals= explainer.shap_values(test_m)
    return shap_vals, test_m.cpu().numpy(), meta_cols


def feature_permutation_importance(model, loader, meta_cols, n_batches=20):
    """
    Permutation feature importance for metadata columns.
    Measures AUC drop when each feature is randomly shuffled.
    """
    model.eval()
    all_imgs, all_metas, all_labels = [], [], []
    for i, batch in enumerate(loader):
        if i >= n_batches:
            break
        all_imgs.append(batch["image"])
        all_metas.append(batch["metadata"])
        all_labels.append(batch["label"])
    imgs   = torch.cat(all_imgs).to(DEVICE)
    metas  = torch.cat(all_metas).to(DEVICE)
    labels = torch.cat(all_labels).numpy()

    with torch.no_grad():
        base_probs = torch.softmax(model(imgs, metas), dim=1)[:,1].cpu().numpy()
    base_auc = roc_auc_score(labels, base_probs)

    importance = {}
    for i, col in enumerate(meta_cols):
        perm   = metas.clone()
        idx    = torch.randperm(metas.size(0))
        perm[:, i] = metas[idx, i]
        with torch.no_grad():
            probs = torch.softmax(model(imgs, perm), dim=1)[:,1].cpu().numpy()
        auc   = roc_auc_score(labels, probs)
        importance[col] = base_auc - auc

    return importance


def feature_ablation(model, loader, meta_cols, n_batches=20):
    """
    Feature ablation study: zero out each metadata feature, measure AUC.
    More conservative than permutation (no data leakage through shuffling).
    """
    model.eval()
    all_imgs, all_metas, all_labels = [], [], []
    for i, batch in enumerate(loader):
        if i >= n_batches:
            break
        all_imgs.append(batch["image"])
        all_metas.append(batch["metadata"])
        all_labels.append(batch["label"])
    imgs   = torch.cat(all_imgs).to(DEVICE)
    metas  = torch.cat(all_metas).to(DEVICE)
    labels = torch.cat(all_labels).numpy()

    with torch.no_grad():
        base_probs = torch.softmax(model(imgs, metas), dim=1)[:,1].cpu().numpy()
    base_auc = roc_auc_score(labels, base_probs)

    ablation = {}
    for i, col in enumerate(meta_cols):
        abl    = metas.clone()
        abl[:, i] = 0.0
        with torch.no_grad():
            probs = torch.softmax(model(imgs, abl), dim=1)[:,1].cpu().numpy()
        auc = roc_auc_score(labels, probs)
        ablation[col] = base_auc - auc

    return ablation


# ─────────────────────────────────────────────────────────────────────────────
# 8. QUANTITATIVE XAI EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

class XAIMetrics:
    """
    Quantitative evaluation of explanation quality.
    Insertion, Deletion, Average Drop, ROAD metrics.
    """

    @staticmethod
    @torch.no_grad()
    def insertion_deletion(model, image, metadata, heatmap, n_steps=10):
        """
        Insertion: progressively insert important pixels → AUC↑ = good map.
        Deletion : progressively delete important pixels → AUC↓ fast = good.
        """
        img_t  = image.unsqueeze(0).to(DEVICE)
        meta_t = metadata.unsqueeze(0).to(DEVICE)
        H, W   = CFG["img_size"], CFG["img_size"]

        flat_idx  = np.argsort(heatmap.ravel())[::-1].copy()   # importance-sorted pixels
        n_pixels  = H * W
        steps     = np.linspace(0, n_pixels, n_steps, dtype=int)

        insertion_scores = []
        deletion_scores  = []
        baseline_blurred = transforms.GaussianBlur(kernel_size=51, sigma=20)(
            image.unsqueeze(0)
        ).to(DEVICE)

        for step in steps:
            # Insertion: start from blurred, add pixels in importance order
            ins_img = baseline_blurred.clone().view(1, 3, -1)
            orig    = img_t.clone().view(1, 3, -1)
            if step > 0:
                ins_img[:, :, flat_idx[:step]] = orig[:, :, flat_idx[:step]]
            ins_img = ins_img.view(1, 3, H, W)
            score_i = torch.softmax(model(ins_img, meta_t), dim=1)[0, 1].item()
            insertion_scores.append(score_i)

            # Deletion: start from original, remove pixels in importance order
            del_img = img_t.clone().view(1, 3, -1)
            if step > 0:
                del_img[:, :, flat_idx[:step]] = 0.0
            del_img = del_img.view(1, 3, H, W)
            score_d = torch.softmax(model(del_img, meta_t), dim=1)[0, 1].item()
            deletion_scores.append(score_d)

        ins_auc = np.trapz(insertion_scores) / (n_steps - 1)
        del_auc = np.trapz(deletion_scores)  / (n_steps - 1)
        return {
            "insertion_auc"  : ins_auc,
            "deletion_auc"   : del_auc,
            "insertion_curve": insertion_scores,
            "deletion_curve" : deletion_scores,
        }

    @staticmethod
    @torch.no_grad()
    def average_drop_increase(model, image, metadata, heatmap, class_idx=None):
        img_t  = image.unsqueeze(0).to(DEVICE)
        meta_t = metadata.unsqueeze(0).to(DEVICE)
        orig   = torch.softmax(model(img_t, meta_t), dim=1)
        if class_idx is None:
            class_idx = orig.argmax(1).item()
        orig_score = orig[0, class_idx].item()

        # --- FIX: Resize heatmap to match image size (224x224) ---
        H, W = CFG["img_size"], CFG["img_size"]
        # Convert numpy heatmap to torch, add batch/channel dims, interpolate, then back to numpy
        h_torch = torch.from_grad(torch.tensor(heatmap)).unsqueeze(0).unsqueeze(0)
        h_resized = F.interpolate(h_torch, size=(H, W), mode='bilinear', align_corners=False).squeeze().numpy()
        # -------------------------------------------------------

        # mask: keep top-50% pixels based on the RESIZED heatmap
        thresh  = np.percentile(h_resized, 50)
        mask_np = (h_resized >= thresh).astype(np.float32)
        mask_t  = torch.tensor(mask_np, device=DEVICE).unsqueeze(0).unsqueeze(0)
        
        masked  = img_t * mask_t

        new_score = torch.softmax(model(masked, meta_t), dim=1)[0, class_idx].item()
        avg_drop  = max(0.0, (orig_score - new_score) / (orig_score + 1e-8)) * 100
        avg_inc   = float(new_score > orig_score)
        return {"avg_drop": avg_drop, "avg_increase": avg_inc,
                "orig_score": orig_score, "masked_score": new_score}


# ─────────────────────────────────────────────────────────────────────────────
# 9. VISUALIZATIONS — PUBLICATION QUALITY
# ─────────────────────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family"     : "DejaVu Serif",
    "font.size"       : 11,
    "axes.titlesize"  : 13,
    "axes.labelsize"  : 11,
    "figure.dpi"      : 150,
    "savefig.dpi"     : 300,
    "savefig.bbox"    : "tight",
})


def plot_training_curves(history, save_path):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle("BioViL-T + FT-Transformer: Training Dynamics", fontsize=15, fontweight="bold")

    metrics = ["loss", "roc_auc", "f1", "accuracy", "sensitivity", "specificity"]
    titles  = ["Loss", "ROC-AUC", "F1 Score", "Accuracy", "Sensitivity", "Specificity"]
    colors  = ["#d62728", "#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd", "#8c564b"]

    for ax, m, t, c in zip(axes.flat, metrics, titles, colors):
        if f"val_{m}" in history:
            ax.plot(history[f"val_{m}"], color=c, linewidth=2, label="Validation")
        if f"train_{m}" in history:
            ax.plot(history[f"train_{m}"], color=c, linewidth=2, linestyle="--",
                    alpha=0.6, label="Train")
        ax.set_title(t)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(t)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


def plot_roc_pr(labels, probs, save_path, model_name="BioViL-T"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"{model_name}: ROC & PR Curves", fontsize=14, fontweight="bold")

    # ROC
    fpr, tpr, _ = roc_curve(labels, probs)
    auc_val     = roc_auc_score(labels, probs)
    ax1.plot(fpr, tpr, color="#1f77b4", lw=2.5, label=f"AUC = {auc_val:.4f}")
    ax1.plot([0,1],[0,1],"--", color="gray", lw=1)
    ax1.fill_between(fpr, tpr, alpha=0.08, color="#1f77b4")
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.set_title("ROC Curve")
    ax1.legend(loc="lower right")
    ax1.grid(True, alpha=0.3)

    # PR
    prec, rec, _ = precision_recall_curve(labels, probs)
    pr_auc       = average_precision_score(labels, probs)
    ax2.plot(rec, prec, color="#d62728", lw=2.5, label=f"AP = {pr_auc:.4f}")
    ax2.fill_between(rec, prec, alpha=0.08, color="#d62728")
    ax2.set_xlabel("Recall")
    ax2.set_ylabel("Precision")
    ax2.set_title("Precision-Recall Curve")
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


def plot_confusion_matrix(labels, preds, save_path, model_name="BioViL-T"):
    cm  = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Non-Mel", "Melanoma"],
                yticklabels=["Non-Mel", "Melanoma"],
                ax=ax, linewidths=0.5)
    ax.set_title(f"{model_name}: Confusion Matrix", fontweight="bold")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


def plot_xai_panel(image_tensor, heatmaps: dict, save_path, image=""):
    """Multi-method XAI visualization panel."""
    n     = len(heatmaps) + 1
    fig   = plt.figure(figsize=(4 * n, 4))
    fig.suptitle(f"XAI Panel — Image: {image}", fontsize=13, fontweight="bold")

    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = image_tensor.permute(1,2,0).cpu().numpy()
    img  = (img * std + mean).clip(0, 1)

    ax   = fig.add_subplot(1, n, 1)
    ax.imshow(img)
    ax.set_title("Original", fontsize=11)
    ax.axis("off")

    for i, (name, hmap) in enumerate(heatmaps.items(), start=2):
        ax = fig.add_subplot(1, n, i)
        overlay = overlay_heatmap(image_tensor, hmap)
        ax.imshow(overlay)
        ax.set_title(name, fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


def plot_feature_importance(importance_dict: dict, title: str, save_path: str):
    df = pd.DataFrame.from_dict(importance_dict, orient="index", columns=["importance"])
    df = df.sort_values("importance", ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(4, len(df) * 0.4)))
    colors = ["#d62728" if v > 0 else "#1f77b4" for v in df["importance"]]
    ax.barh(df.index, df["importance"], color=colors, edgecolor="white")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("AUC Change (drop = important)")
    ax.axvline(0, color="black", lw=0.8)
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


def plot_tsne(features, labels, save_path, model_name="BioViL-T"):
    print("[INFO] Computing t-SNE...")
    tsne   = TSNE(n_components=2, perplexity=30, random_state=SEED)
    embed  = tsne.fit_transform(features[:500])   # limit for speed
    labs   = labels[:500]
    fig, ax = plt.subplots(figsize=(8, 6))
    for cls_idx, (name, col) in enumerate(zip(["Non-Melanoma","Melanoma"],
                                               ["#1f77b4","#d62728"])):
        mask = labs == cls_idx
        ax.scatter(embed[mask,0], embed[mask,1], c=col, label=name,
                   alpha=0.6, s=20, edgecolors="none")
    ax.set_title(f"{model_name}: t-SNE Feature Embeddings", fontweight="bold")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


def plot_gate_weights(gates, labels, save_path):
    """Visualize modality gate weights: how much image vs metadata the model relies on."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Modality Gate Weights Analysis", fontsize=13, fontweight="bold")

    for cls_idx, name in enumerate(["Non-Melanoma", "Melanoma"]):
        mask = labels == cls_idx
        g    = gates[mask]
        axes[0].hist(g[:, 0], bins=20, alpha=0.6,
                     label=f"{name} (image gate)", density=True)
        axes[1].hist(g[:, 1], bins=20, alpha=0.6,
                     label=f"{name} (meta gate)", density=True)

    for ax, title in zip(axes, ["Image Gate Weight", "Metadata Gate Weight"]):
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Gate Value")
        ax.set_ylabel("Density")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


def plot_insertion_deletion(metrics_dict, save_path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle("Quantitative XAI: Insertion & Deletion Curves", fontweight="bold")

    for name, m in metrics_dict.items():
        ax1.plot(m["insertion_curve"], label=f"{name} (AUC={m['insertion_auc']:.3f})")
        ax2.plot(m["deletion_curve"],  label=f"{name} (AUC={m['deletion_auc']:.3f})")

    ax1.set_title("Insertion Curve (↑ better)")
    ax1.set_xlabel("Fraction of pixels revealed")
    ax1.set_ylabel("Predicted score")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.set_title("Deletion Curve (↓ faster = better)")
    ax2.set_xlabel("Fraction of pixels deleted")
    ax2.set_ylabel("Predicted score")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


# ─────────────────────────────────────────────────────────────────────────────
# 10. ABLATION STUDIES
# ─────────────────────────────────────────────────────────────────────────────
def run_ablation_study(base_model, loaders, meta_cols):
    """
    Ablation studies to quantify contribution of each architectural component.
    Each ablation removes or replaces ONE component at a time.
    """
    results = {}
    criterion = FocalLoss(CFG["focal_alpha"], CFG["focal_gamma"],
                           CFG["label_smoothing"]).to(DEVICE)

    # ── Baseline ─────────────────────────────────────────────────────────
    print("\n[ABLATION] Baseline (full model)...")
    metrics = evaluate(base_model, loaders["val"], criterion, DEVICE)
    results["Full Model"] = metrics

    # ── Ablation 1: Image-only (remove metadata) ──────────────────────────
    class ImageOnlyModel(nn.Module):
        def __init__(self, full_model):
            super().__init__()
            self.vision_enc = full_model.vision_enc
            self.head       = nn.Sequential(
                nn.Linear(CFG["img_embed_dim"], 128),
                nn.GELU(), nn.Dropout(0.3),
                nn.Linear(128, 2)
            )
        def forward(self, img, meta):
            cls, _ = self.vision_enc(img)
            return self.head(cls)

    img_only = ImageOnlyModel(base_model).to(DEVICE)
    metrics  = evaluate(img_only, loaders["val"], criterion, DEVICE)
    results["Image Only (No Metadata)"] = metrics
    print(f"  Image-Only AUC: {metrics['roc_auc']:.4f}")

    # ── Ablation 2: Metadata-only ──────────────────────────────────────────
    class MetaOnlyModel(nn.Module):
        def __init__(self, full_model):
            super().__init__()
            self.meta_enc = full_model.meta_enc
            self.head     = nn.Sequential(
                nn.Linear(CFG["ft_d_token"], 64),
                nn.GELU(), nn.Dropout(0.3),
                nn.Linear(64, 2)
            )
        def forward(self, img, meta):
            cls, _ = self.meta_enc(meta)
            return self.head(cls)

    meta_only = MetaOnlyModel(base_model).to(DEVICE)
    metrics   = evaluate(meta_only, loaders["val"], criterion, DEVICE)
    results["Metadata Only (No Image)"] = metrics
    print(f"  Meta-Only AUC: {metrics['roc_auc']:.4f}")

    # ── Ablation 3: Replace FT-Transformer with MLP ───────────────────────
    class MLPMetaModel(nn.Module):
        def __init__(self, full_model, n_meta):
            super().__init__()
            self.vision_enc = full_model.vision_enc
            self.meta_mlp   = nn.Sequential(
                nn.Linear(n_meta, 128), nn.ReLU(),
                nn.Linear(128, 128),   nn.ReLU()
            )
            self.head       = nn.Linear(CFG["img_embed_dim"] + 128, 2)
        def forward(self, img, meta):
            img_f, _ = self.vision_enc(img)
            meta_f   = self.meta_mlp(meta)
            return self.head(torch.cat([img_f, meta_f], dim=-1))

    n_meta = len(meta_cols)
    mlp_model = MLPMetaModel(base_model, n_meta).to(DEVICE)
    metrics   = evaluate(mlp_model, loaders["val"], criterion, DEVICE)
    results["FT-Transformer → MLP"] = metrics
    print(f"  MLP-Meta AUC: {metrics['roc_auc']:.4f}")

    # ── Ablation 4: Remove Stochastic Depth ──────────────────────────────
    # (not re-trainable easily in one eval pass; report from training if available)
    results["No Stochastic Depth"] = {"roc_auc": float("nan"), "note": "Requires retraining"}

    # ── Report ────────────────────────────────────────────────────────────
    print("\n[ABLATION RESULTS]")
    print(f"{'Component':<35} {'AUC':>8} {'F1':>8} {'Acc':>8}")
    print("-" * 60)
    for name, m in results.items():
        auc = m.get("roc_auc", float("nan"))
        f1  = m.get("f1", float("nan"))
        acc = m.get("accuracy", float("nan"))
        print(f"{name:<35} {auc:>8.4f} {f1:>8.4f} {acc:>8.4f}")

    return results


def plot_ablation(ablation_results, save_path):
    df = pd.DataFrame([
        {"Component": k, "ROC-AUC": v.get("roc_auc", float("nan")),
         "F1": v.get("f1", float("nan")), "Accuracy": v.get("accuracy", float("nan"))}
        for k, v in ablation_results.items()
    ]).set_index("Component")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Ablation Study: Component Contribution Analysis",
                 fontsize=14, fontweight="bold")

    for ax, col, color in zip(axes, ["ROC-AUC", "F1", "Accuracy"],
                               ["#1f77b4", "#2ca02c", "#ff7f0e"]):
        vals = df[col].dropna()
        bars = ax.barh(vals.index, vals.values, color=color, alpha=0.8, edgecolor="white")
        ax.set_title(col, fontweight="bold")
        ax.set_xlim(vals.min() * 0.95, 1.0)
        ax.set_xlabel("Score")
        ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
        ax.grid(True, axis="x", alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")


# ─────────────────────────────────────────────────────────────────────────────
# 11. ERROR ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────
def error_analysis(eval_results, datasets, save_path):
    """
    Analyze false positives and false negatives.
    Identify challenging cases for clinical interpretation.
    """
    labels = eval_results["labels"]
    preds  = eval_results["preds"]
    probs  = eval_results["probs"]
    ids    = eval_results["ids"]

    fp_idx = np.where((preds == 1) & (labels == 0))[0]
    fn_idx = np.where((preds == 0) & (labels == 1))[0]
    tp_idx = np.where((preds == 1) & (labels == 1))[0]
    tn_idx = np.where((preds == 0) & (labels == 0))[0]

    print(f"\n[ERROR ANALYSIS]")
    print(f"  True Positives  (Mel → Mel)    : {len(tp_idx)}")
    print(f"  True Negatives  (Non → Non)    : {len(tn_idx)}")
    print(f"  False Positives (Non → Mel)    : {len(fp_idx)}  ← over-diagnosis risk")
    print(f"  False Negatives (Mel → Non)    : {len(fn_idx)}  ← missed diagnosis risk")

    # High-confidence errors
    if len(fp_idx):
        fp_confs = probs[fp_idx]
        print(f"\n  High-confidence FP (prob>0.9): "
              f"{(fp_confs>0.9).sum()} cases")
    if len(fn_idx):
        fn_confs = 1 - probs[fn_idx]
        print(f"  High-confidence FN (prob<0.1): "
              f"{(fn_confs>0.9).sum()} cases")

    # Uncertainty analysis
    uncertainty = np.abs(probs - 0.5)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle("Error Analysis: Prediction Confidence Distribution",
                 fontweight="bold")

    for cls_idx, name in enumerate(["Non-Melanoma", "Melanoma"]):
        axes[0].hist(probs[labels == cls_idx], bins=30, alpha=0.6, label=name, density=True)
    axes[0].set_title("Predicted Probability Distribution by Class")
    axes[0].set_xlabel("P(Melanoma)")
    axes[0].set_ylabel("Density")
    axes[0].legend()
    axes[0].axvline(0.5, color="red", lw=1.5, linestyle="--", label="Decision threshold")
    axes[0].grid(True, alpha=0.3)

    # Confidence histogram
    correct = (preds == labels)
    axes[1].hist(uncertainty[correct],  bins=20, alpha=0.7, label="Correct",   color="#2ca02c", density=True)
    axes[1].hist(uncertainty[~correct], bins=20, alpha=0.7, label="Incorrect", color="#d62728", density=True)
    axes[1].set_title("Prediction Uncertainty |p - 0.5|")
    axes[1].set_xlabel("Distance from Decision Boundary")
    axes[1].set_ylabel("Density")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"[SAVED] {save_path}")

    return {"fp_count": len(fp_idx), "fn_count": len(fn_idx),
            "tp_count": len(tp_idx), "tn_count": len(tn_idx)}


# ─────────────────────────────────────────────────────────────────────────────
# 12.  MAIN EXECUTION PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
# 12.  MAIN EXECUTION PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("=" * 70)
    print("  BioViL-T + FT-Transformer + Lesion-Conditioned Sparse Attention")
    print("  Multimodal Explainable AI for Skin Cancer Classification")
    print("=" * 70)

    # ── 1. Metadata Column Detection ──────────────────────────────────────
    # Detect all relevant metadata columns automatically from the CSV
    csv_check = TRAIN_DIR / "train.csv"
    if csv_check.exists():
        df_check = pd.read_csv(csv_check)
        # Exclude administrative and target columns
        exclude = ["image", "isic_id", "patient_id", "year", "class", "image_fixed", "__img_path", "label"]
        # Select columns that are numeric (int/float) and not in the exclude list
        META_COLS = [c for c in df_check.columns 
                     if c not in exclude and df_check[c].dtype in [np.float64, np.float32, np.int64]]
        print(f"[INFO] Automatically detected {len(META_COLS)} clinical features.")
    else:
        print("[ERROR] Could not find train.csv to detect features.")
        return

    n_meta = len(META_COLS)

    # ── 2. Build DataLoaders ──────────────────────────────────────────────
    loaders, datasets = build_dataloaders(META_COLS)
    if not loaders:
        print("[ERROR] No data found. Check dataset paths.")
        return

    # ── 3. Build Model ───────────────────────────────────────────────────
    model = BioViLFTModel(n_meta_features=n_meta, n_classes=2).to(DEVICE)
    
    # ── 4. Check for Existing Checkpoint (Load before Compile) ──────────
    ckpt_path = OUTPUT_DIR / "best_model.pth"
    skip_training = False

    if ckpt_path.exists():
        print(f"\n[INFO] Found existing checkpoint at {ckpt_path}. Skipping training.")
        # weights_only=False required for numpy scalars in best_auc
        state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        
        # TURBO KEY FIXER: Remove '_orig_mod.' prefix from keys if they exist
        new_state_dict = {}
        for k, v in state["model"].items():
            name = k.replace("_orig_mod.", "") 
            new_state_dict[name] = v
        
        model.load_state_dict(new_state_dict)
        print(f"[INFO] Successfully loaded Best Val AUC: {state['best_auc']:.4f}")
        skip_training = True
    else:
        print("[INFO] No checkpoint found. Proceeding to training mode.")

    # ── 5. Apply Turbo Compilation (Graph Optimization) ─────────────────
    try:
        model = torch.compile(model)
        print("[INFO] Torch model compiled successfully.")
    except Exception as e:
        print(f"[INFO] Torch compile skipped: {e}")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[INFO] Trainable parameters: {n_params:,}")

    # ── 6. Train only if checkpoint was not found ────────────────────────
    if not skip_training:
        if "train" in loaders and "val" in loaders:
            print("\n[TRAINING] Starting optimized training pipeline...")
            history = train_pipeline(model, loaders)
            plot_training_curves(history, OUTPUT_DIR / "training_curves.png")
        else:
            print("[ERROR] No checkpoint found and training loaders are missing.")
            return

    # ── 7. Full Evaluation ───────────────────────────────────────────────
    eval_split = "test" if "test" in loaders else "val"
    print(f"\n[EVALUATION] Running on {eval_split} split...")
    eval_res = full_evaluation(model, loaders[eval_split], eval_split)

    plot_roc_pr(eval_res["labels"], eval_res["probs"],
                OUTPUT_DIR / "roc_pr_curves.png")
    plot_confusion_matrix(eval_res["labels"], eval_res["preds"],
                          OUTPUT_DIR / "confusion_matrix.png")
    plot_tsne(eval_res["features"], eval_res["labels"],
              OUTPUT_DIR / "tsne_embeddings.png")
    plot_gate_weights(eval_res["gates"], eval_res["labels"],
                      OUTPUT_DIR / "gate_weights.png")

    # ── 8. XAI — Explainable AI Panel ────────────────────────────────────
    if eval_split in datasets:
        print("\n[XAI] Generating explanations...")
        
        # TURBO FIX: Unwrap the compiled model for XAI hooks to work properly
        model_xai = model._orig_mod if hasattr(model, '_orig_mod') else model
        model_xai.eval()
        
        sample_ds = datasets[eval_split]

        # Pick indices for one Melanoma and one Non-Melanoma sample
        mel_indices = np.where(sample_ds.labels == 1)[0]
        nmel_indices = np.where(sample_ds.labels == 0)[0]
        
        target_samples = []
        if len(mel_indices) > 0: target_samples.append((int(mel_indices[0]), "melanoma"))
        if len(nmel_indices) > 0: target_samples.append((int(nmel_indices[0]), "non_melanoma"))

        for sample_idx, label_name in target_samples:
            sample   = sample_ds[sample_idx]
            img_t    = sample["image"]
            meta_t   = sample["metadata"]
            img_id   = sample["image_id"]

            # Target layer must be from the UNWRAPPED model for hooks to register
            target_layer = model_xai.vision_enc.backbone.blocks[-1].norm1

            print(f"  Processing {label_name.upper()} (ID: {img_id})...")
            
            # Initialize heatmaps dictionary
            heatmaps = {}
            try:
                # GradCAM
                gcam = GradCAM(model_xai, target_layer)
                heatmaps["GradCAM"] = gcam(img_t, meta_t)
                
                # GradCAM++
                gcpp = GradCAMPlusPlus(model_xai, target_layer)
                heatmaps["GradCAM++"] = gcpp(img_t, meta_t)
                
                # Integrated Gradients (Turbo: 12 steps)
                ig = IntegratedGradients(model_xai, n_steps=12)
                heatmaps["IntGrad"] = ig(img_t, meta_t)
                
                # Occlusion Sensitivity
                occ = OcclusionSensitivity(model_xai, patch_size=28, stride=14)
                heatmaps["Occlusion"] = occ(img_t, meta_t)
                
                # SmoothGrad
                sg = SmoothGrad(model_xai, n_samples=15)
                heatmaps["SmoothGrad"] = sg(img_t, meta_t)
                
                # Attention Rollout
                ar = AttentionRollout(model_xai)
                heatmaps["AttnRollout"] = ar(img_t, meta_t)
                
            except Exception as e:
                print(f"    [WARN] XAI method failed: {e}")

            # Visualization & Quantitative metrics
            if heatmaps:
                plot_xai_panel(img_t, heatmaps, 
                               OUTPUT_DIR / f"xai_panel_{label_name}.png", 
                               image=img_id)

                # Quantitative XAI (Insertion/Deletion/Drop)
                xai_quant = {}
                for name, hmap in heatmaps.items():
                    ins_del = XAIMetrics.insertion_deletion(model_xai, img_t, meta_t, hmap)
                    avg_dr  = XAIMetrics.average_drop_increase(model_xai, img_t, meta_t, hmap)
                    xai_quant[name] = {**ins_del, **avg_dr}
                    print(f"    [{name}] InsAUC={ins_del['insertion_auc']:.3f} | DelAUC={ins_del['deletion_auc']:.3f}")

                plot_insertion_deletion(xai_quant, OUTPUT_DIR / f"insertion_deletion_{label_name}.png")

        # ── 9. Metadata Feature Importance ───────────────────────────────
        print("\n[XAI] Computing metadata feature importance...")
        perm_imp = feature_permutation_importance(model_xai, loaders[eval_split], META_COLS, n_batches=10)
        plot_feature_importance(perm_imp, "Permutation Feature Importance (AUC Drop)",
                                str(OUTPUT_DIR / "perm_importance.png"))

        abl_imp  = feature_ablation(model_xai, loaders[eval_split], META_COLS, n_batches=10)
        plot_feature_importance(abl_imp, "Feature Ablation Importance (AUC Drop)",
                                str(OUTPUT_DIR / "ablation_importance.png"))

        # Metadata SHAP Beeswarm
        shap_res = compute_shap_values(model_xai, loaders[eval_split], META_COLS, n_bg=20)
        if shap_res[0] is not None:
            import shap
            shap_vals, test_m, cols = shap_res
            fig, ax = plt.subplots(figsize=(10, 6))
            shap.summary_plot(shap_vals[1], test_m, feature_names=cols, show=False, plot_type="dot")
            plt.title("SHAP Beeswarm — Metadata (Melanoma Class)", fontweight="bold")
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / "shap_beeswarm.png")
            plt.close()

    # ── 10. Final Reporting ──────────────────────────────────────────────
    print("\n[ERROR ANALYSIS]...")
    error_analysis(eval_res, datasets, OUTPUT_DIR / "error_analysis.png")

    if "val" in loaders:
        print("\n[ABLATION] Running ablation studies...")
        # Note: model_xai used here to avoid compiled graph overhead in small forward passes
        abl_res = run_ablation_study(model_xai, loaders, META_COLS)
        plot_ablation(abl_res, OUTPUT_DIR / "ablation_study.png")

    # Save metrics JSON
    with open(OUTPUT_DIR / "evaluation_metrics.json", "w") as f:
        json.dump({k: float(v) for k, v in eval_res["metrics"].items()}, f, indent=2)
    
    print(f"\n[INFO] All outputs saved to: {OUTPUT_DIR}")
    print("\n" + "=" * 70)
    print("  FINAL TEST METRICS")
    print("=" * 70)
    for k, v in eval_res["metrics"].items():
        print(f"  {k:20s}: {v:.4f}")
    print("=" * 70)


if __name__ == "__main__":
    main()

[INFO] Device: cuda
[INFO] Config loaded: {'img_size': 224, 'batch_size': 32, 'num_workers': 4, 'num_epochs': 1, 'lr': 0.0003, 'weight_decay': 0.0001, 'warmup_epochs': 1, 'grad_clip': 1.0, 'label_smoothing': 0.1, 'ema_decay': 0.99, 'focal_alpha': 0.25, 'focal_gamma': 2.0, 'dropout': 0.3, 'stoch_depth': 0.1, 'num_classes': 2, 'grad_accum': 2, 'ft_d_token': 128, 'ft_n_heads': 8, 'ft_n_layers': 3, 'ft_ffn_factor': 1.3333333333333333, 'sparse_topk': 8, 'img_embed_dim': 768}
  BioViL-T + FT-Transformer + Lesion-Conditioned Sparse Attention
  Multimodal Explainable AI for Skin Cancer Classification
[INFO] Automatically detected 29 clinical features.
[Dataset:train] 11941 samples found.
[Dataset:val] 2559 samples found.
[Dataset:test] 2559 samples found.

[INFO] Found existing checkpoint at outputs/notebook1_biovil/best_model.pth. Skipping training.
[INFO] Successfully loaded Best Val AUC: 0.6259
[INFO] Torch model compiled successfully.
[INFO] Trainable parameters: 87,915,778

[EVALUATION] R

AttributeError: module 'torch' has no attribute 'from_grad'